# Training Linear Probes: What Does Each Layer Know?

A **linear probe** is a simple classifier (logistic regression) trained on a model's internal activations. If a probe can classify some property with high accuracy, that property is **linearly represented** at that layer.

This is one of the most powerful tools for understanding *what information* lives where in a transformer.

This notebook covers:
1. What probes are and why they work
2. Extracting activations from specific layers
3. Training probes to detect syntactic and semantic features
4. Comparing probe accuracy across layers
5. Best practices and pitfalls

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt

import irtk
from irtk.probes import LinearProbe, train_linear_probe, ProbeTrainResult
from irtk.datasets import to_tokens

print("Loading GPT-2 Small...")
model = irtk.HookedTransformer.from_pretrained("gpt2")
tokenizer = model.tokenizer
print(f"Loaded: {model.cfg.n_layers} layers, d_model={model.cfg.d_model}")

## 1. The Idea Behind Probes

A transformer's residual stream at layer `l` and position `p` is a vector in $\mathbb{R}^{d_{\text{model}}}$. A linear probe asks:

> Is there a **direction** in this space that separates examples with property X from those without it?

If yes, the model has learned to linearly represent that property at that layer. This is a much stronger claim than just "the information is somewhere in the activations" (which is trivially true for any representation).

## 2. Experiment: Probing for Number Parity

Let's test whether GPT-2 linearly represents whether a number is even or odd. We'll:
1. Feed the model prompts containing numbers
2. Extract residual stream activations at the number position
3. Train a probe to classify even vs odd

In [ ]:
# Generate data: sentences with numbers
np.random.seed(42)
numbers = list(range(1, 100))  # 1 to 99
templates = [
    "The number {} is",
    "There are {} items",
    "She counted {} apples",
    "He has {} coins",
]

prompts = []
labels = []  # 0 = even, 1 = odd

for num in numbers:
    template = templates[num % len(templates)]
    prompt = template.format(num)
    prompts.append(prompt)
    labels.append(num % 2)

labels = np.array(labels)
print(f"Created {len(prompts)} prompts ({sum(labels == 0)} even, {sum(labels == 1)} odd)")
print(f"\nExamples:")
for i in range(4):
    print(f"  {prompts[i]!r} -> {'odd' if labels[i] else 'even'}")

In [ ]:
# Extract residual stream activations at the number position for each layer
# We'll look at the position where the number token is

layer_activations = {l: [] for l in range(model.cfg.n_layers)}
valid_labels = []

for i, prompt in enumerate(prompts):
    tokens = to_tokens(prompt, tokenizer=tokenizer)
    _, cache = model.run_with_cache(tokens)
    
    # The number is typically at position 2-3 ("The number XX"),
    # but let's find it by looking for the last numeric token
    tok_strs = [tokenizer.decode([int(t)]) for t in tokens]
    
    # Use position after the number (where the model processes it)
    # For simplicity, use a fixed position where the number would be
    num_pos = min(3, len(tokens) - 1)  # approximate position
    
    for l in range(model.cfg.n_layers):
        resid = cache[("resid_post", l)]  # [seq_len, d_model]
        layer_activations[l].append(np.array(resid[num_pos]))
    valid_labels.append(labels[i])

# Stack into arrays
for l in layer_activations:
    layer_activations[l] = jnp.array(np.stack(layer_activations[l]))
valid_labels = jnp.array(valid_labels, dtype=jnp.int32)

print(f"Extracted activations for {len(valid_labels)} examples at {model.cfg.n_layers} layers")
print(f"Activation shape per layer: {layer_activations[0].shape}")

In [ ]:
# Train a probe at each layer
results_by_layer = {}

for l in range(model.cfg.n_layers):
    result = train_linear_probe(
        layer_activations[l], valid_labels,
        n_classes=2, epochs=50, lr=1e-3, verbose=False,
    )
    results_by_layer[l] = result
    if l % 3 == 0:
        print(f"Layer {l:2d}: best val acc = {result.best_val_acc:.3f} (epoch {result.best_epoch})")

print(f"...")
print(f"Layer {model.cfg.n_layers-1:2d}: best val acc = {results_by_layer[model.cfg.n_layers-1].best_val_acc:.3f}")

In [ ]:
# Plot probe accuracy across layers
layer_accs = [results_by_layer[l].best_val_acc for l in range(model.cfg.n_layers)]

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(range(model.cfg.n_layers), layer_accs, 'o-', markersize=6)
ax.axhline(y=0.5, color='gray', linestyle='--', alpha=0.5, label='Random chance')
ax.set_xlabel("Layer")
ax.set_ylabel("Best Validation Accuracy")
ax.set_title("Linear Probe: Number Parity (Even/Odd) by Layer")
ax.set_xticks(range(0, model.cfg.n_layers, 2))
ax.set_ylim(0.35, 1.05)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

best_layer = np.argmax(layer_accs)
print(f"\nBest layer for parity: Layer {best_layer} ({layer_accs[best_layer]:.3f})")

## 3. Experiment: Probing for Token Identity

Let's probe something simpler: can we predict what the current token is from the residual stream? This should be easy at early layers (near the embedding) and might degrade at later layers as representations become more abstract.

In [ ]:
# Generate data: random sentences, probe which token is at a given position
sentences = [
    "The cat sat on the mat and looked at the fish",
    "A quick brown fox jumped over the lazy sleeping dog",
    "The president gave a long speech about the economy today",
    "She walked to the store and bought some fresh bread",
    "The old man sat on the bench reading his book",
    "They went to the park and played with the ball",
    "The sun was shining bright on the green grassy hill",
    "He opened the door and saw the beautiful garden outside",
    "The children were playing in the yard after school ended",
    "A bird flew over the tall building in the city",
]

# Extract activations and token IDs for a specific position
probe_pos = 3  # position to probe
act_by_layer = {l: [] for l in range(model.cfg.n_layers)}
token_ids = []

for sent in sentences:
    tokens = to_tokens(sent, tokenizer=tokenizer)
    if len(tokens) <= probe_pos:
        continue
    _, cache = model.run_with_cache(tokens)
    token_ids.append(int(tokens[probe_pos]))
    for l in range(model.cfg.n_layers):
        act_by_layer[l].append(np.array(cache[("resid_post", l)][probe_pos]))

# Since we have few unique tokens, let's make this a binary classification:
# is it a common word (" the", " a", etc.) or not?
common_tokens = set(tokenizer.encode(w)[0] for w in [" the", " a", " an", " on", " in"])
binary_labels = jnp.array([1 if t in common_tokens else 0 for t in token_ids], dtype=jnp.int32)

print(f"Probing position {probe_pos} across {len(sentences)} sentences")
print(f"Class balance: {int(sum(binary_labels == 1))} common tokens, {int(sum(binary_labels == 0))} other")
print(f"Tokens at pos {probe_pos}: {[tokenizer.decode([t]) for t in token_ids]}")

## 4. Training Loop Details

Let's look at the training dynamics of a probe in more detail.

In [ ]:
# Train a probe on the middle layer with verbose output
mid_layer = model.cfg.n_layers // 2

# Use the number parity data (larger dataset)
result = train_linear_probe(
    layer_activations[mid_layer], valid_labels,
    n_classes=2, epochs=100, lr=1e-3, verbose=False,
)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Loss curves
ax1.plot(result.train_losses, label='Train loss')
ax1.plot(result.val_losses, label='Val loss')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Cross-entropy Loss')
ax1.set_title(f'Training Curves (Layer {mid_layer})')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Accuracy curves
ax2.plot(result.train_accs, label='Train acc')
ax2.plot(result.val_accs, label='Val acc')
ax2.axhline(y=0.5, color='gray', linestyle='--', alpha=0.5)
ax2.axhline(y=result.best_val_acc, color='green', linestyle=':', alpha=0.5, label=f'Best val: {result.best_val_acc:.3f}')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.set_title(f'Accuracy Curves (Layer {mid_layer})')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nBest validation accuracy: {result.best_val_acc:.3f} at epoch {result.best_epoch}")

## 5. Probing Different Activations

We've been probing `resid_post` (the residual stream after each layer). But we can also probe:
- `resid_pre`: before the layer
- `attn_out`: just the attention output
- `mlp_out`: just the MLP output

This tells us whether the information comes from attention, MLPs, or was already present.

In [ ]:
# For a specific layer, compare probe accuracy on different activations
target_layer = best_layer  # use the best layer from the parity experiment

# Re-collect activations for different hook points at this layer
hook_names = {
    "resid_pre": f"blocks.{target_layer}.hook_resid_pre",
    "attn_out": f"blocks.{target_layer}.hook_attn_out",
    "mlp_out": f"blocks.{target_layer}.hook_mlp_out",
    "resid_post": f"blocks.{target_layer}.hook_resid_post",
}

hook_activations = {name: [] for name in hook_names}

for i, prompt in enumerate(prompts):
    tokens = to_tokens(prompt, tokenizer=tokenizer)
    _, cache = model.run_with_cache(tokens)
    num_pos = min(3, len(tokens) - 1)
    
    for name, hook in hook_names.items():
        act = cache[hook]  # [seq_len, d_model]
        hook_activations[name].append(np.array(act[num_pos]))

# Train probes
print(f"Probing Layer {target_layer} with different activation types:")
print("=" * 50)
hook_accs = {}
for name in hook_names:
    acts = jnp.array(np.stack(hook_activations[name]))
    result = train_linear_probe(
        acts, valid_labels, n_classes=2, epochs=50, lr=1e-3, verbose=False,
    )
    hook_accs[name] = result.best_val_acc
    print(f"  {name:12s}: val acc = {result.best_val_acc:.3f}")

# Plot comparison
fig, ax = plt.subplots(figsize=(8, 5))
names = list(hook_accs.keys())
accs = [hook_accs[n] for n in names]
colors = ['#4C72B0', '#55A868', '#C44E52', '#8172B2']
ax.bar(names, accs, color=colors)
ax.axhline(y=0.5, color='gray', linestyle='--', alpha=0.5)
ax.set_ylabel('Best Validation Accuracy')
ax.set_title(f'Parity Probe: Different Activations at Layer {target_layer}')
ax.set_ylim(0.35, 1.05)
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

## Key Takeaways

1. **Linear probes test for linear representations**: If a probe achieves high accuracy, the feature is linearly decodable at that layer.

2. **Layer matters**: Different features are best represented at different layers. Early layers capture syntax, later layers capture semantics.

3. **Which activation to probe**: `resid_post` contains everything. Probing `attn_out` vs `mlp_out` tells you *which component* computes the feature.

4. **Pitfalls**:
   - Small datasets can give misleading results (always use train/val split)
   - High accuracy could mean the probe is overfitting
   - A probe might pick up on spurious correlations rather than the intended feature
   - Non-linear probes find more information but are harder to interpret

### API Reference

```python
from irtk.probes import (
    LinearProbe,         # eqx.Module: d_model -> n_classes
    RegressionProbe,     # For continuous targets
    train_linear_probe,  # Full training loop with train/val split
    ProbeTrainResult,    # Results dataclass with losses, accs, best model
)

# Quick start:
result = train_linear_probe(
    activations,     # [n_samples, d_model]
    labels,          # [n_samples] integer labels
    n_classes=2,
    epochs=100,
    lr=1e-3,
)
print(f"Best val accuracy: {result.best_val_acc}")
```